# 📖 Notebook 4: Stream Processing Patterns

Welcome to the final notebook in our Kafka deep dive! In this notebook, we'll explore
**stream processing** — one of the most powerful things you can do with Kafka.

## What You'll Learn

- The difference between **batch processing** and **stream processing**
- How to **filter**, **transform**, and **enrich** messages in real-time
- How to **aggregate** data using **tumbling windows**
- Real-world stream processing use cases

> **Note:** "Kafka Streams" is a Java library built into the Kafka ecosystem.
> In Python, there is no official Kafka Streams equivalent. Instead, we use the
> **consumer/producer API** from `confluent-kafka` to build stream processing
> patterns manually. This is a great way to understand what's happening under the hood!

## 🛠️ Setup

Before running this notebook, make sure:

1. **Kafka is running** via Docker Compose:
   ```bash
   cd 03-technologies/messaging/kafka && docker compose up -d
   ```

2. **Select the `.venv` kernel** in VS Code's kernel picker (top-right of the notebook).
   If the kernel doesn't appear, reload VS Code (`Cmd+Shift+P` → "Reload Window").

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
import json, time, random
from collections import defaultdict
from datetime import datetime

KAFKA_CONFIG = {'bootstrap.servers': 'localhost:9092'}

admin = AdminClient(KAFKA_CONFIG)
try:
    metadata = admin.list_topics(timeout=5)
    print("✅ Connected to Kafka!")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   Run: cd 03-technologies/messaging/kafka && docker compose up -d")

def create_topic(name, partitions=1):
    t = NewTopic(name, num_partitions=partitions, replication_factor=1)
    fs = admin.create_topics([t])
    for n, f in fs.items():
        try:
            f.result()
            print(f"✅ Created topic '{n}'")
        except Exception as e:
            print(f"ℹ️ {n}: {e}")

## 🔄 Batch vs Stream Processing

There are two main ways to process data:

### Batch Processing
Collect data over a period of time, then process **all of it at once**.

Think of it like doing laundry once a week — you wait until you have a full load,
then wash everything together.

**Examples:** nightly sales reports, weekly analytics summaries, monthly billing.

### Stream Processing
Process **each piece of data as it arrives**, in real-time.

Think of it like a conveyor belt in a factory — each item is inspected, stamped,
or sorted the moment it passes by. You don't wait for the belt to stop.

**Examples:** real-time dashboards, fraud detection, live sports updates, ad click counting.

```
Batch:   [collect...collect...collect] → [PROCESS ALL] → result
Stream:  event → process → result, event → process → result, ...
```

### Common Stream Processing Patterns

| Pattern | What It Does |
|---------|-------------|
| **Filter** | Keep only messages matching a condition |
| **Map / Transform** | Change the shape or content of each message |
| **Enrich** | Add external context to each message |
| **Aggregate** | Combine multiple messages into a summary |
| **Window** | Aggregate over fixed time periods |

Let's implement each one! 🚀

---

## Pattern 1: Filter 🔍

**Filter** means: only keep messages that match a condition, drop the rest.

Imagine you're watching a World Cup match and your phone buzzes for *every* event:
throw-ins, corners, fouls... But you only care about **goals** and **red cards**.

A filter does exactly that — it lets the important stuff through and silently
drops everything else.

```
Input (raw-events):        Output (important-events):
  goal         ────────►     goal ✅
  corner       ────── ✗      (dropped)
  foul         ────── ✗      (dropped)
  red_card     ────────►     red_card ✅
  throw_in     ────── ✗      (dropped)
```

Let's generate some sample World Cup events first, then filter them.

In [ ]:
# Generate sample World Cup events
create_topic('raw-events', partitions=2)

producer = Producer(KAFKA_CONFIG)

event_types = ['goal', 'yellow_card', 'red_card', 'substitution', 'corner', 'foul', 'offside', 'throw_in']
matches = ['Brazil vs Germany', 'Argentina vs France', 'Spain vs Japan']
players = ['Neymar', 'Müller', 'Messi', 'Mbappé', 'Pedri', 'Kubo', 'Kane', 'Mané']

events = []
for i in range(20):
    event = {
        'id': i,
        'match': random.choice(matches),
        'type': random.choice(event_types),
        'player': random.choice(players),
        'minute': random.randint(1, 90),
        'timestamp': time.time(),
    }
    events.append(event)
    producer.produce('raw-events', key=event['match'].encode(), value=json.dumps(event).encode())

producer.flush()
print(f"✅ Published 20 random World Cup events")
print(f"\nSample events:")
for e in events[:5]:
    print(f"  {e['match']} | {e['type']} | {e['player']} (min {e['minute']})")

In [ ]:
# FILTER: Only keep goals and red cards
create_topic('important-events')

consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'filter-demo', 'auto.offset.reset': 'earliest'})
consumer.subscribe(['raw-events'])

output_producer = Producer(KAFKA_CONFIG)

print("🔍 FILTER: Keeping only goals and red cards")
print("=" * 50)

filtered_count = 0
total_count = 0
empty = 0

while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    total_count += 1

    event = json.loads(msg.value().decode())

    # FILTER: only important events
    if event['type'] in ('goal', 'red_card'):
        output_producer.produce('important-events', value=msg.value())
        filtered_count += 1
        emoji = "⚽" if event['type'] == 'goal' else "🟥"
        print(f"  {emoji} KEPT: {event['match']} — {event['type']} by {event['player']} (min {event['minute']})")
    else:
        pass  # silently dropped

consumer.close()
output_producer.flush()

print(f"\n📊 Filtered {total_count} events → {filtered_count} important events")
print(f"   {total_count - filtered_count} events dropped (corners, fouls, etc.)")

---

## Pattern 2: Map / Transform 🔄

**Map** means: transform each message into a **different shape**.

The input and output are 1-to-1 — every message in produces exactly one message out,
but the output looks different from the input.

**Example:** Convert raw match events into user-friendly push notifications.

```
Input (raw event):                      Output (notification):
  {type: 'goal', player: 'Messi'}  →     {text: '⚽ GOAL! Messi scores!', priority: 'high'}
  {type: 'red_card', player: 'Kane'} →   {text: '🟥 RED CARD! Kane sent off!', priority: 'medium'}
```

In [ ]:
# MAP: Transform important events into user notifications
create_topic('notifications')

consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'map-demo', 'auto.offset.reset': 'earliest'})
consumer.subscribe(['important-events'])

output_producer = Producer(KAFKA_CONFIG)

print("🔄 MAP: Transforming events into user notifications")
print("=" * 50)

empty = 0
count = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    count += 1

    event = json.loads(msg.value().decode())

    # MAP: transform into a notification
    if event['type'] == 'goal':
        text = f"⚽ GOAL! {event['player']} scores for {event['match'].split(' vs ')[0]} in minute {event['minute']}!"
    elif event['type'] == 'red_card':
        text = f"🟥 RED CARD! {event['player']} is sent off in {event['match']}!"
    else:
        text = f"📢 {event['type']}: {event['player']} in {event['match']}"

    notification = {
        'text': text,
        'match': event['match'],
        'priority': 'high' if event['type'] == 'goal' else 'medium',
        'created_at': time.time(),
    }

    output_producer.produce('notifications', value=json.dumps(notification).encode())
    print(f"  {notification['text']}")

consumer.close()
output_producer.flush()
print(f"\n📊 Transformed {count} events into notifications")

---

## Pattern 3: Aggregate (Counting) 📊

**Aggregate** means: combine multiple messages into a **summary**.

Unlike Filter and Map (which work on one message at a time), aggregation needs to
**remember state** — it keeps a running tally as new messages arrive.

**Example:** Count how many events of each type happened in each match.

```
Messages arriving one by one:          Aggregated result:
  Brazil vs Germany: goal              Brazil vs Germany:
  Brazil vs Germany: corner              goal: 2
  Argentina vs France: foul              corner: 1
  Brazil vs Germany: goal              Argentina vs France:
  Argentina vs France: goal              foul: 1
                                         goal: 1
```

> **Important:** We keep the aggregation state in memory (a Python dictionary).
> In production, you'd use a database or Kafka Streams' built-in state stores.

In [ ]:
# AGGREGATE: Count events per match
create_topic('match-stats')

consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'aggregate-demo', 'auto.offset.reset': 'earliest'})
consumer.subscribe(['raw-events'])

# In-memory aggregation state
match_stats = defaultdict(lambda: defaultdict(int))

print("📊 AGGREGATE: Counting events per match")
print("=" * 50)

empty = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0

    event = json.loads(msg.value().decode())
    match_stats[event['match']][event['type']] += 1

consumer.close()

print("\n📋 Match Statistics:")
print("-" * 50)
for match, stats in sorted(match_stats.items()):
    print(f"\n  {match}:")
    for event_type, count in sorted(stats.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"    {event_type:<15} {bar} ({count})")

# Publish aggregated results
output_producer = Producer(KAFKA_CONFIG)
for match, stats in match_stats.items():
    result = {'match': match, 'stats': dict(stats), 'total_events': sum(stats.values())}
    output_producer.produce('match-stats', key=match.encode(), value=json.dumps(result).encode())
output_producer.flush()

print(f"\n✅ Published aggregated stats for {len(match_stats)} matches")

---

## Pattern 4: Tumbling Window 🪟

Real-time systems often need to aggregate data **over time periods**.

A **tumbling window** collects data for a fixed time period (e.g., 10 seconds),
computes a result, then **starts a fresh window**. Windows don't overlap — each
event belongs to exactly one window.

```
Time:    |---10s---|---10s---|---10s---|
Clicks:  [5 clicks] [3 clicks] [7 clicks]
Output:     → 5        → 3        → 7
```

**Example:** Count ad clicks per 10-second window.

This is how real-time dashboards work — they show you "clicks in the last 10 seconds"
or "orders in the last minute" by using tumbling windows.

> **How it works:** Each event has a timestamp. We calculate which window it belongs to
> using integer division: `window_start = (timestamp // window_size) * window_size`.
> All events in the same window get grouped together.

In [ ]:
# TUMBLING WINDOW: Count ad clicks per 10-second window
create_topic('ad-clicks', partitions=2)
create_topic('click-counts')

# First, generate some click data spread over time
producer = Producer(KAFKA_CONFIG)

print("📨 Generating ad click events (simulating 30 seconds of data)...")

ads = ['nike-shoe', 'apple-iphone', 'tesla-model3']
click_data = []
base_time = time.time()
for i in range(30):
    click = {
        'ad_id': random.choice(ads),
        'user_id': f'user-{random.randint(1, 100)}',
        'timestamp': base_time + i,  # 1 click per second
    }
    click_data.append(click)
    producer.produce('ad-clicks', key=click['ad_id'].encode(), value=json.dumps(click).encode())

producer.flush()
print(f"✅ Generated {len(click_data)} clicks over 30 simulated seconds\n")

# Now process with tumbling windows
consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'window-demo', 'auto.offset.reset': 'earliest'})
consumer.subscribe(['ad-clicks'])

output_producer = Producer(KAFKA_CONFIG)

WINDOW_SIZE_SECONDS = 10  # 10-second tumbling windows

# Window state: {window_start_time: {ad_id: click_count}}
windows = {}

print("🪟 TUMBLING WINDOW: Counting clicks per 10-second window")
print("=" * 55)

empty = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0

    click = json.loads(msg.value().decode())
    event_time = click['timestamp']

    # Determine which window this click belongs to
    window_start = int(event_time // WINDOW_SIZE_SECONDS) * WINDOW_SIZE_SECONDS

    if window_start not in windows:
        windows[window_start] = defaultdict(int)

    windows[window_start][click['ad_id']] += 1

consumer.close()

# Display window results
print("\n📊 Window Results:")
for window_start in sorted(windows.keys()):
    offset = int(window_start - base_time)
    counts = windows[window_start]
    total = sum(counts.values())
    print(f"\n  Window [{offset}s - {offset + WINDOW_SIZE_SECONDS}s]: {total} total clicks")
    for ad, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        bar = "█" * cnt
        print(f"    {ad:<20} {bar} ({cnt})")

    # Publish window result
    result = {'window_start': window_start, 'window_size_sec': WINDOW_SIZE_SECONDS, 'counts': dict(counts), 'total': total}
    output_producer.produce('click-counts', value=json.dumps(result).encode())

output_producer.flush()
print(f"\n✅ Published {len(windows)} window aggregations")

---

## Pattern 5: Enrich (Join with External Data) 🔗

**Enrich** means: add **context** to each message by looking up external data.

Think of it like a factory worker who stamps each box with extra info:
the box arrives with just a product ID, and the worker looks up the product name,
price, and category from a catalog, then sticks a label on the box.

**Example:** Each ad click only has an `ad_id`. We enrich it with campaign metadata
(campaign name, budget, region) from a lookup table.

```
Input click:                  Lookup table:                 Enriched output:
  {ad_id: 'nike-shoe'}   +   nike-shoe → Summer Sale   =   {ad_id: 'nike-shoe',
                                          $50K, Global       campaign: 'Summer Sale',
                                                             budget: 50000,
                                                             region: 'Global'}
```

> **In production**, the lookup table could be a database, a REST API, or even
> another Kafka topic (this is called a "stream-table join").

In [ ]:
# ENRICH: Add campaign metadata to click events

# Simulated "database" of ad campaigns
ad_campaigns = {
    'nike-shoe': {'campaign': 'Summer Sale 2024', 'budget': 50000, 'region': 'Global'},
    'apple-iphone': {'campaign': 'iPhone 16 Launch', 'budget': 100000, 'region': 'US+EU'},
    'tesla-model3': {'campaign': 'EV Revolution', 'budget': 75000, 'region': 'US'},
}

create_topic('enriched-clicks')

consumer = Consumer({**KAFKA_CONFIG, 'group.id': 'enrich-demo', 'auto.offset.reset': 'earliest'})
consumer.subscribe(['ad-clicks'])

output_producer = Producer(KAFKA_CONFIG)

print("🔗 ENRICH: Adding campaign metadata to click events")
print("=" * 55)

empty = 0
count = 0
while empty < 3:
    msg = consumer.poll(2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    count += 1

    click = json.loads(msg.value().decode())

    # ENRICH: look up campaign info
    campaign_info = ad_campaigns.get(click['ad_id'], {})
    enriched = {**click, **campaign_info}

    output_producer.produce('enriched-clicks', value=json.dumps(enriched).encode())

    if count <= 5:
        print(f"  Click on '{click['ad_id']}' → enriched with campaign '{campaign_info.get('campaign', 'N/A')}'")

if count > 5:
    print(f"  ... and {count - 5} more")

consumer.close()
output_producer.flush()
print(f"\n📊 Enriched {count} click events with campaign metadata")

---

## 🌍 Real-World Stream Processing

Here's how the patterns we learned map to real-world use cases:

| Pattern | Example | Input → Output |
|---------|---------|---------------|
| **Filter** | Spam detection | All emails → Non-spam emails |
| **Map** | Format conversion | Raw logs → Structured JSON |
| **Aggregate** | Analytics | Page views → Views per URL |
| **Window** | Real-time metrics | Click stream → Clicks per minute |
| **Enrich** | Context adding | Orders → Orders + customer info |

### Production Stream Processing Frameworks

In this notebook, we built everything manually with `confluent-kafka`'s consumer/producer API.
This is great for learning, but in production you'd typically use a dedicated framework:

- **Kafka Streams** (Java) — Built into Kafka. Great if you're in the Java ecosystem.
- **Apache Flink** — Powerful, distributed stream processing engine. Supports Python.
- **Apache Spark Streaming** — Batch-style stream processing (micro-batches).

These frameworks handle the hard parts for you: fault tolerance, state management,
exactly-once processing, and scaling across multiple machines.

---

## 📝 Key Takeaways

1. **Stream processing** = processing data as it arrives (vs **batch** = collect first, process later)
2. **Core patterns**: Filter, Map, Aggregate, Window, Enrich
3. Kafka's **consumer/producer API** can implement these patterns manually in Python
4. **Tumbling windows** aggregate data over fixed time periods (no overlap)
5. **Enrichment** joins streaming data with external context (databases, APIs, lookup tables)
6. For **production** use: Kafka Streams (Java), Apache Flink, or Spark Streaming

---

## 🎉 Congratulations!

You've completed the **Kafka Deep Dive** series! Here's a recap of everything you've learned:

| Notebook | Topic | What You Learned |
|----------|-------|------------------|
| **01** | Kafka Basics | Topics, producers, consumers, partitions |
| **02** | Consumer Groups | Parallel processing, rebalancing, offsets |
| **03** | Exactly-Once Semantics | Delivery guarantees, idempotent producers, transactions |
| **04** | Stream Processing | Filter, Map, Aggregate, Window, Enrich |
| **05** | Production Best Practices | Schema validation, DLQ, compaction, replication, batching |

### Next Steps

- 🔍 **Explore Kafka UI** at [http://localhost:8080](http://localhost:8080) to see all the topics
  we created across all notebooks
- 📚 Read the [Kafka documentation](https://kafka.apache.org/documentation/)
- 🧪 Try modifying the patterns — what happens if you chain a filter → map → aggregate?
- 🚀 Explore production frameworks like [Apache Flink](https://flink.apache.org/) or
  [Kafka Streams](https://kafka.apache.org/documentation/streams/)

Happy streaming! 🚀